# RAG with Reranking - Complete Guide

Packages used (NO `langchain-community`):
- `langchain-groq` for Groq LLM
- `langchain-huggingface` for Embeddings
- `langchain-chroma` for Vector store
- `rank-bm25` for BM25 (directly)
- `sentence-transformers` for Cross-Encoder

## Approaches Covered:
1. Baseline RAG (vector similarity)
2. BM25 Retrieval (keyword)
3. Cross-Encoder Reranking (neural)
4. LLM as Reranker (Groq)
5. Results Comparison

In [1]:
# Install first-party LangChain packages (no langchain-community)
!pip install -q langchain langchain-groq langchain-huggingface langchain-chroma
!pip install -q sentence-transformers rank-bm25 pandas matplotlib


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, time, re
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for nbconvert
import matplotlib.pyplot as plt
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from rank_bm25 import BM25Okapi

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY', 'your_groq_api_key_here')
print('All imports done!')

All imports done!


In [3]:
docs_text = [
    'Transformers use self-attention and were introduced in Attention is All You Need in 2017. They revolutionized NLP.',
    'BERT is a pre-trained NLP model by Google. It reads text bidirectionally. Used for sentiment analysis and NER.',
    'GPT is a language model by OpenAI that generates text by predicting next tokens. GPT-4 powers ChatGPT.',
    'RAG (Retrieval Augmented Generation) retrieves relevant documents then uses an LLM to generate answers. Reduces hallucination.',
    'Vector databases store embeddings for semantic search. Popular ones: Chroma, Pinecone, Weaviate, Faiss.',
    'Embeddings are numerical vector representations of text. Similar texts have similar vectors. sentence-transformers creates them.',
    'BM25 is a classical keyword retrieval algorithm based on term frequency. Fast and requires no ML model.',
    'Cross-encoders score query-document pairs simultaneously. More accurate but slower than bi-encoders. Used for reranking.',
    'Reranking in RAG re-scores initially retrieved documents so only the most relevant ones reach the LLM for generation.',
    'LLaMA is an open-source LLM by Meta that runs locally on consumer hardware. LLaMA 3 is widely used.',
    'Fine-tuning adapts a pre-trained model to specific tasks with less data. LoRA and QLoRA are efficient methods.',
    'Self-attention computes relationships between all tokens in a sequence. Multi-head attention uses multiple heads in parallel.',
]

documents = [Document(page_content=t, metadata={'id': i}) for i, t in enumerate(docs_text)]
print(f'Created {len(documents)} documents')
print('Sample:', documents[0].page_content)

Created 12 documents
Sample: Transformers use self-attention and were introduced in Attention is All You Need in 2017. They revolutionized NLP.


In [4]:
# langchain-huggingface (not community)
print('Loading embeddings...')
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

# langchain-chroma (not community)
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name='rag_demo'
)
print('Vector store ready with', vectorstore._collection.count(), 'docs')

Loading embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store ready with 12 docs


In [5]:
llm = ChatGroq(model='qwen/qwen3.8-27b', temperature=0)

test_query = 'What is reranking in RAG and why is it useful?'

qa_prompt = PromptTemplate(
    template='Use the context to answer briefly.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:',
    input_variables=['context', 'question']
)

print('LLM ready!')
print('Test Query:', test_query)

LLM ready!
Test Query: What is reranking in RAG and why is it useful?


---
## Approach 1: Baseline RAG (Vector Similarity)

In [6]:
vector_retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

start = time.time()
baseline_docs = vector_retriever.invoke(test_query)
baseline_time = time.time() - start

print('BASELINE RAG - Retrieved Docs:')
for i, doc in enumerate(baseline_docs):
    print(f'  [Doc {i+1}] {doc.page_content[:110]}')
print(f'Time: {baseline_time:.3f}s')

BASELINE RAG - Retrieved Docs:
  [Doc 1] Reranking in RAG re-scores initially retrieved documents so only the most relevant ones reach the LLM for gene
  [Doc 2] RAG (Retrieval Augmented Generation) retrieves relevant documents then uses an LLM to generate answers. Reduce
  [Doc 3] Cross-encoders score query-document pairs simultaneously. More accurate but slower than bi-encoders. Used for 
Time: 0.029s


In [7]:
ctx = '\n\n'.join([d.page_content for d in baseline_docs])
formatted_prompt = qa_prompt.format(context=ctx, question=test_query)
baseline_answer = llm.invoke([HumanMessage(content=formatted_prompt)]).content
print('Baseline RAG Answer:')
print(baseline_answer)

Baseline RAG Answer:
Reranking in RAG re-scores initially retrieved documents to ensure only the most relevant ones reach the LLM for generation, which helps improve answer quality.


---
## Approach 2: BM25 Retrieval (Keyword-based)

Uses `rank_bm25` package directly - zero community dependency.

In [8]:
# Build BM25 index
tokenized_corpus = [doc.page_content.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_retrieve(query, k=3):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_k_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [(documents[i], float(scores[i])) for i in top_k_idx]

start = time.time()
bm25_results = bm25_retrieve(test_query, k=3)
bm25_time = time.time() - start

bm25_docs = [doc for doc, score in bm25_results]

print('BM25 - Retrieved Docs:')
for i, (doc, score) in enumerate(bm25_results):
    print(f'  [Doc {i+1}] Score={score:.4f} | {doc.page_content[:100]}')
print(f'Time: {bm25_time:.3f}s')

BM25 - Retrieved Docs:
  [Doc 1] Score=4.2742 | Reranking in RAG re-scores initially retrieved documents so only the most relevant ones reach the LL
  [Doc 2] Score=3.1486 | BERT is a pre-trained NLP model by Google. It reads text bidirectionally. Used for sentiment analysi
  [Doc 3] Score=2.6416 | Transformers use self-attention and were introduced in Attention is All You Need in 2017. They revol
Time: 0.000s


In [9]:
ctx = '\n\n'.join([d.page_content for d in bm25_docs])
formatted_prompt = qa_prompt.format(context=ctx, question=test_query)
bm25_answer = llm.invoke([HumanMessage(content=formatted_prompt)]).content
print('BM25 Answer:')
print(bm25_answer)

BM25 Answer:
Reranking in RAG re-scores initially retrieved documents to ensure only the most relevant ones reach the LLM for generation.


---
## Approach 3: Cross-Encoder Reranking (Neural Reranking Model)

In [10]:
from sentence_transformers import CrossEncoder

print('Loading Cross-Encoder...')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('Cross-Encoder ready!')

Loading Cross-Encoder...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-Encoder ready!


In [11]:
# Step 1: Retrieve 6 candidates via vector search
candidate_retriever = vectorstore.as_retriever(search_kwargs={'k': 6})
candidate_docs = candidate_retriever.invoke(test_query)
print(f'Step 1: Retrieved {len(candidate_docs)} candidates')

# Step 2: Rerank with cross-encoder
start = time.time()
pairs = [(test_query, doc.page_content) for doc in candidate_docs]
scores = cross_encoder.predict(pairs)

scored = sorted(zip(scores, candidate_docs), key=lambda x: x[0], reverse=True)
reranked_docs = [doc for s, doc in scored[:3]]
reranked_scores = [s for s, doc in scored[:3]]
cross_encoder_time = time.time() - start

print('\nCross-Encoder Top 3 after reranking:')
for i, (doc, s) in enumerate(zip(reranked_docs, reranked_scores)):
    print(f'  [Doc {i+1}] Score={s:.4f} | {doc.page_content[:100]}')
print(f'Time: {cross_encoder_time:.3f}s')

Step 1: Retrieved 6 candidates

Cross-Encoder Top 3 after reranking:
  [Doc 1] Score=4.7171 | Reranking in RAG re-scores initially retrieved documents so only the most relevant ones reach the LL
  [Doc 2] Score=-3.0293 | RAG (Retrieval Augmented Generation) retrieves relevant documents then uses an LLM to generate answe
  [Doc 3] Score=-6.6235 | Cross-encoders score query-document pairs simultaneously. More accurate but slower than bi-encoders.
Time: 0.051s


In [12]:
ctx = '\n\n'.join([d.page_content for d in reranked_docs])
formatted_prompt = qa_prompt.format(context=ctx, question=test_query)
cross_encoder_answer = llm.invoke([HumanMessage(content=formatted_prompt)]).content
print('Cross-Encoder Reranking Answer:')
print(cross_encoder_answer)

Cross-Encoder Reranking Answer:
Reranking in RAG re-scores initially retrieved documents to ensure only the most relevant ones reach the LLM for generation, which helps improve answer quality.


---
## Approach 4: LLM as Reranker (Groq)

In [13]:
candidates = candidate_retriever.invoke(test_query)
print(f'Retrieved {len(candidates)} candidates')

start = time.time()
docs_str = ''
for i, doc in enumerate(candidates):
    docs_str += f'[Document {i+1}]\n{doc.page_content}\n\n'

rerank_prompt = (
    'You are a relevance scorer.\n'
    f'Query: {test_query}\n\n'
    'Rank TOP 3 most relevant document numbers. Reply ONLY with numbers like: 4, 1, 6\n\n'
    f'{docs_str}Top 3 (most relevant first):'
)

ranking_text = llm.invoke([HumanMessage(content=rerank_prompt)]).content.strip()
print('LLM Ranking Response:', ranking_text)

nums = [int(n) for n in re.findall(r'\d+', ranking_text) if 1 <= int(n) <= len(candidates)]
llm_reranked_docs = [candidates[i-1] for i in nums[:3]]
llm_rerank_time = time.time() - start

print('\nLLM Reranking Top 3:')
for i, doc in enumerate(llm_reranked_docs):
    print(f'  [Doc {i+1}] {doc.page_content[:110]}')
print(f'Time: {llm_rerank_time:.3f}s')

Retrieved 6 candidates


LLM Ranking Response: 1, 3, 2

LLM Reranking Top 3:
  [Doc 1] Reranking in RAG re-scores initially retrieved documents so only the most relevant ones reach the LLM for gene
  [Doc 2] Cross-encoders score query-document pairs simultaneously. More accurate but slower than bi-encoders. Used for 
  [Doc 3] RAG (Retrieval Augmented Generation) retrieves relevant documents then uses an LLM to generate answers. Reduce
Time: 0.247s


In [14]:
ctx = '\n\n'.join([d.page_content for d in llm_reranked_docs])
formatted_prompt = qa_prompt.format(context=ctx, question=test_query)
llm_reranked_answer = llm.invoke([HumanMessage(content=formatted_prompt)]).content
print('LLM Reranker Answer:')
print(llm_reranked_answer)

LLM Reranker Answer:
Reranking in RAG re-scores initially retrieved documents to ensure only the most relevant ones reach the LLM for generation, which helps reduce hallucination.


---
## Final Results Comparison

In [15]:
results = [
    {'name': 'Baseline RAG',  'method': 'Vector Similarity', 'time': baseline_time,      'answer': baseline_answer},
    {'name': 'BM25',          'method': 'Keyword BM25',      'time': bm25_time,          'answer': bm25_answer},
    {'name': 'Cross-Encoder', 'method': 'Neural Reranker',   'time': cross_encoder_time, 'answer': cross_encoder_answer},
    {'name': 'LLM Reranker',  'method': 'Groq LLM Scorer',  'time': llm_rerank_time,    'answer': llm_reranked_answer},
]

print('=' * 65)
print('FINAL RESULTS COMPARISON')
print('=' * 65)
for r in results:
    print(f"\n--- {r['name']} ({r['method']}) ---")
    print(f"  Time   : {r['time']:.3f}s")
    ans = r['answer']
    print(f"  Answer : {ans[:200]}" + ('...' if len(ans) > 200 else ''))
print('\n' + '=' * 65)

FINAL RESULTS COMPARISON

--- Baseline RAG (Vector Similarity) ---
  Time   : 0.029s
  Answer : Reranking in RAG re-scores initially retrieved documents to ensure only the most relevant ones reach the LLM for generation, which helps improve answer quality.

--- BM25 (Keyword BM25) ---
  Time   : 0.000s
  Answer : Reranking in RAG re-scores initially retrieved documents to ensure only the most relevant ones reach the LLM for generation.

--- Cross-Encoder (Neural Reranker) ---
  Time   : 0.051s
  Answer : Reranking in RAG re-scores initially retrieved documents to ensure only the most relevant ones reach the LLM for generation, which helps improve answer quality.

--- LLM Reranker (Groq LLM Scorer) ---
  Time   : 0.247s
  Answer : Reranking in RAG re-scores initially retrieved documents to ensure only the most relevant ones reach the LLM for generation, which helps reduce hallucination.



In [16]:
df = pd.DataFrame({
    'Approach':  ['Baseline RAG', 'BM25', 'Cross-Encoder', 'LLM Reranker'],
    'Package':   ['langchain-chroma', 'rank-bm25', 'sentence-transformers', 'langchain-groq'],
    'Time (s)':  [round(baseline_time,3), round(bm25_time,3), round(cross_encoder_time,3), round(llm_rerank_time,3)],
    'Accuracy':  ['Medium', 'Low', 'High', 'High'],
    'Speed':     ['Fast', 'Very Fast', 'Medium', 'Slow'],
    'API Cost':  ['None', 'None', 'None', 'Groq API'],
})
print('Comparison Table:')
print(df.to_string(index=False))

Comparison Table:
     Approach               Package  Time (s) Accuracy     Speed API Cost
 Baseline RAG      langchain-chroma     0.029   Medium      Fast     None
         BM25             rank-bm25     0.000      Low Very Fast     None
Cross-Encoder sentence-transformers     0.051     High    Medium     None
 LLM Reranker        langchain-groq     0.247     High      Slow Groq API


In [17]:
labels  = ['Baseline\nRAG', 'BM25', 'Cross-\nEncoder', 'LLM\nReranker']
times_v = [baseline_time, bm25_time, cross_encoder_time, llm_rerank_time]
colors  = ['#3498db', '#f1c40f', '#e67e22', '#2ecc71']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('RAG Reranking Methods - Comparison', fontsize=13, fontweight='bold')

# Chart 1: Time comparison
bars = ax1.bar(labels, times_v, color=colors, edgecolor='black')
ax1.set_title('Retrieval + Reranking Time (s)')
ax1.set_ylabel('Seconds')
for bar, t in zip(bars, times_v):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{t:.3f}s', ha='center', fontsize=9)

# Chart 2: Accuracy vs Speed
accuracy = [3, 2, 5, 4.5]
speed    = [4, 5, 3, 1]
x = range(len(labels))
w = 0.35
ax2.bar([i - w/2 for i in x], accuracy, w, label='Accuracy', color='#9b59b6', alpha=0.85)
ax2.bar([i + w/2 for i in x], speed,    w, label='Speed',    color='#1abc9c', alpha=0.85)
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels)
ax2.set_ylim(0, 6.5)
ax2.set_title('Accuracy vs Speed (score/5)')
ax2.set_ylabel('Score')
ax2.legend()

plt.tight_layout()
plt.savefig('reranking_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved!')

Chart saved!


C:\Users\asus\AppData\Local\Temp\ipykernel_22748\205698002.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Key Takeaways

| Method | Package | Pros | Cons |
|--------|---------|------|------|
| Baseline RAG | `langchain-chroma` | Fast, simple | May miss keywords |
| BM25 | `rank-bm25` | Very fast, no ML | No semantic understanding |
| Cross-Encoder | `sentence-transformers` | Most accurate | Slower |
| LLM Reranker | `langchain-groq` | No extra model | Slow, uses API credits |

**Best Practice**: BM25 + Vector Hybrid retrieval then Cross-Encoder reranking then LLM answer